# Provider Phase 1 Source Profile

## tl;dr

This notebook is a reader-facing companion to the automated Provider source profiler. It reads the executed evidence under `outputs/provider_phase1/2026-07-29`; it does not recalculate CMS risk-adjusted results or convert suppression values to zero.

## Context & Methods

### Key Assumptions

- Inputs are public, aggregate CMS provider files, not patient-level claims or EHR data.
- Facility ID is a string.
- Source release, measurement period, and fiscal year are separate concepts.
- The module `src.quality.profile_provider` is the executable source of truth for profiling logic.

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
evidence_dir = project_root / "outputs" / "provider_phase1" / "2026-07-29"
assert evidence_dir.is_dir(), evidence_dir

## Data

Load the bounded summary, QA, period, and measure evidence produced by the executed profiler.

In [ ]:
dataset_summary = pd.read_csv(evidence_dir / "dataset_summary.csv", dtype=str)
quality_checks = pd.read_csv(evidence_dir / "quality_checks.csv", dtype=str)
measurement_periods = pd.read_csv(evidence_dir / "measurement_periods.csv", dtype=str)
measure_catalog = pd.read_csv(evidence_dir / "measure_catalog.csv", dtype=str)

dataset_summary[["dataset_id", "row_count", "column_count", "candidate_key", "suppression_token_count"]]

## Results

In [ ]:
status_counts = quality_checks["status"].value_counts().rename_axis("status").reset_index(name="checks")
nonpass = quality_checks.loc[quality_checks["status"] != "PASS"]
status_counts, nonpass

In [ ]:
measurement_periods.sort_values(["dataset_id", "measurement_start_date", "measurement_end_date"])

## Takeaways

- Review the executed narrative in `outputs/provider_phase1/2026-07-29/profiling_report.md`.
- A clean Phase 1 run does not by itself approve the later provider model, benchmark joins, SQL views, or Power BI semantic model.
- Any future CMS release must receive a new immutable snapshot and be compared with the accepted schema baseline.